In [1]:
!pip install lightgbm -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Ready")

Mounted at /content/drive
✅ Ready


In [3]:
DATA_DIR = '/content/drive/MyDrive/머신러닝/data'  # ⚠️ 본인 경로!
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

Train: (21693, 54), Test: (15004, 53)


In [4]:
def feature_engineering(df):
    """FE 1차 + X_01 제거 (검증된 최종 버전)"""
    df = df.copy()

    # X_01 제거
    if 'X_01' in df.columns:
        df = df.drop(columns=['X_01'])

    sensors = [c for c in df.columns if c.startswith('X_')]

    # 행 단위 통계
    df['row_mean']   = df[sensors].mean(axis=1)
    df['row_std']    = df[sensors].std(axis=1)
    df['row_min']    = df[sensors].min(axis=1)
    df['row_max']    = df[sensors].max(axis=1)
    df['row_range']  = df['row_max'] - df['row_min']
    df['row_skew']   = df[sensors].skew(axis=1)
    df['row_median'] = df[sensors].median(axis=1)

    # 핵심 센서 통계
    core_sensors = ['X_19', 'X_37', 'X_40', 'X_11']
    df['core_mean'] = df[core_sensors].mean(axis=1)
    df['core_std']  = df[core_sensors].std(axis=1)
    df['core_max']  = df[core_sensors].max(axis=1)
    df['core_min']  = df[core_sensors].min(axis=1)

    # 핵심 센서 페어 조합
    df['X19_X37_ratio'] = df['X_19'] / (df['X_37'] + 1e-6)
    df['X19_X40_ratio'] = df['X_19'] / (df['X_40'] + 1e-6)
    df['X19_X11_ratio'] = df['X_19'] / (df['X_11'] + 1e-6)
    df['X19_X37_diff']  = df['X_19'] - df['X_37']
    df['X19_X40_diff']  = df['X_19'] - df['X_40']
    df['X37_X40_diff']  = df['X_37'] - df['X_40']

    return df

# 적용
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)
print(f"FE 후: Train {train_fe.shape}, Test {test_fe.shape}")

FE 후: Train (21693, 70), Test (15004, 69)


In [5]:
# ============================================
# 데이터 준비
# ============================================
X = train_fe.drop(columns=['ID', 'target']).values
y = train_fe['target'].values
X_test = test_fe.drop(columns=['ID']).values

# 정규화 (LightGBM은 필수 아니지만, 일관성 위해)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

NUM_CLASSES = 21
print(f"피처 수: {X.shape[1]}")

# ============================================
# LightGBM 5-fold 학습 (OOF + Test 예측)
# ============================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(X), NUM_CLASSES))
test_probs = np.zeros((len(X_test), NUM_CLASSES))
fold_scores = []

print("\n" + "="*60)
print("🚀 LightGBM 5-fold 학습")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000,         # 좀 더 많이
        learning_rate=0.03,         # 좀 더 작게 (안정성)
        num_leaves=63,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False)])

    # OOF + Test 예측
    oof_probs[val_idx] = model.predict_proba(X_val)
    test_probs += model.predict_proba(X_test) / 5

    # Fold 점수
    fold_pred = oof_probs[val_idx].argmax(axis=1)
    f1 = f1_score(y_val, fold_pred, average='macro')
    fold_scores.append(f1)
    print(f"  Fold {fold}: Macro-F1 = {f1:.4f}")

# OOF 전체 점수
oof_pred = oof_probs.argmax(axis=1)
oof_f1 = f1_score(y, oof_pred, average='macro')
print(f"\n📊 OOF Macro-F1: {oof_f1:.4f} (평균: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f})")

피처 수: 68

🚀 LightGBM 5-fold 학습
  Fold 0: Macro-F1 = 0.8069
  Fold 1: Macro-F1 = 0.8052
  Fold 2: Macro-F1 = 0.8046
  Fold 3: Macro-F1 = 0.8008
  Fold 4: Macro-F1 = 0.8027

📊 OOF Macro-F1: 0.8041 (평균: 0.8040 ± 0.0021)


In [6]:
# ============================================
# Argmax 후처리 → 제출 파일 (검증 결과 반영)
# ============================================
preds = test_probs.argmax(axis=1)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = f'{DATA_DIR}/submission_lightgbm.csv'
submission.to_csv(output_path, index=False)

print(f"✅ Saved: {output_path}")
print(f"\n📊 클래스별 예측 개수:")
print(pd.Series(preds).value_counts().sort_index())

✅ Saved: /content/drive/MyDrive/머신러닝/data/submission_lightgbm.csv

📊 클래스별 예측 개수:
0      864
1      688
2      416
3      913
4      725
5      454
6      710
7      440
8      991
9      717
10     733
11     652
12    1250
13     682
14     700
15     844
16     562
17     711
18     668
19     573
20     711
Name: count, dtype: int64
